In [ ]:
# 1. Install the missing libraries
!pip install groq nest_asyncio pandas -q

# 2. Re-import and initialize
import nest_asyncio
from groq import Groq

nest_asyncio.apply()

# Note: Wrapped the key in quotes to avoid a NameError
GROQ_API_KEY = "API KEY"
client = Groq(api_key=GROQ_API_KEY)

# 3. Check available models
try:
    models = client.models.list()
    print("Successfully connected! Available models:")
    for model in models.data:
        print(f" - {model.id}")
except Exception as e:
    print(f"Connection failed: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.0 MB/s eta 0:00:00
Successfully connected! Available models:
 - llama-3.1-8b-instant
 - openai/gpt-oss-120b
 - meta-llama/llama-prompt-guard-2-22m
 - meta-llama/llama-prompt-guard-2-86m
 - llama-3.3-70b-versatile
 - whisper-large-v3
 - groq/compound
 - meta-llama/llama-4-scout-17b-16e-instruct
 - canopylabs/orpheus-arabic-saudi
 - whisper-large-v3-turbo
 - allam-2-7b
 - openai/gpt-oss-20b
 - qwen/qwen3-32b
 - groq/compound-mini
 - openai/gpt-oss-safeguard-20b
 - canopylabs/orpheus-v1-english


In [ ]:
# ============================================================================
# CELL 1 — Imports & Configuration
# ============================================================================
!pip install groq nest_asyncio -q

import json, re, time, logging
import nest_asyncio
import pandas as pd
from groq import Groq

nest_asyncio.apply()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

GROQ_API_KEY = "API_KEY"
client       = Groq(api_key=GROQ_API_KEY)
MODEL_ID     = "openai/gpt-oss-120b"

INPUT_PATH  = "Input_Data_GEC.xlsx"
OUTPUT_PATH = "GPT_Responses_Data_GEC.xlsx"

SYSTEM_PROMPT = """You are an expert in Odia (ଓଡ଼ିଆ) linguistics specializing in Grammatical Error Detection (GED) and Grammatical Error Correction (GEC). Analyze the given text and identify all linguistic errors with high precision.

## Categories (apply in this priority order — assign only the highest-priority matching category per span)

1. Script Normalization — Unicode-level encoding errors where the character sequence is wrong even if the rendering looks similar to the correct form. Includes: nukta misplacement relative to a base consonant, incorrect virama usage in conjunct formation, vowel sign decomposition errors where two combining marks are used instead of a single precomposed one, and ZWNJ/ZWJ presence or absence errors affecting ligature formation or consonant cluster boundaries. Also includes inappropriate consonant conjunct or cluster formation caused by the absence of a ZWNJ, where two adjacent consonants incorrectly merge into a ligature instead of remaining as distinct units.

2. Spelling & Typographical Errors — Unicode encoding is correct but the wrong characters are used. Includes: confusion between short and long vowel signs, substitution among phonetically similar consonants, missing or extra characters within a word, and incorrect word boundaries (two words merged or one word split). [*Note: if the error is in how characters are encoded rather than which characters are chosen, classify as Script Normalization instead.]

3. Grammatical Errors — Morphosyntactic errors where the script and spelling are correct. Includes: wrong verb tense, aspect, or inflectional form; agreement failure; incorrect postpositional case marker; wrong conjunction for the syntactic context; incorrect voice; light verb misuse; word order violations; wrong copular constructions; and missing punctuation that is grammatically required.

4. Code-Mixing / Wrong Language — Odia text containing elements from a different language or script. Includes: Roman-script words or abbreviations where an Odia equivalent exists, non-Odia numeral systems where Odia digits are standard, characters from another Indic script embedded in Odia text, and loanword phrases where a standard Odia term is available.

5. Correct Sentence / No Errors — Return this as a single entry if no errors are found.

## Rules
- Priority: If a span qualifies under multiple categories, assign only the highest-priority matching category (1 is highest).
- Single error assumption: Each input sentence contains exactly one error, which may span a single character, a word, or a longer phrase (e.g., a word order violation). Identify the single best error span and assign it to exactly one category. If multiple categories seem applicable, rank them by the priority order above and assign the highest-ranking one.
- Span overlap: Report the one erroneous span only. Do not report sub-spans of the same error separately.
- Correction scope: Make minimum necessary edits only — do not reorder, paraphrase, or add content beyond what fixing the identified error requires.
- Binary flag: Set "has_errors" to true if an error is found, and false if the sentence is correct. When false, "errors" must be an empty array and "corrected_sentence" must be identical to the input.

## Output
Return ONLY valid JSON. No markdown, no text outside the JSON.

{
  "has_errors": <true | false>,
  "errors": [
    {
      "error_span": "<exact erroneous text>",
      "category": "<one of the five categories>",
      "description": "<technical explanation>"
    }
  ],
  "corrected_sentence": "<fully corrected sentence, or identical to input if no errors>"
}"""

In [ ]:
# ============================================================================
# CELL 2 — API Call & Response Parsing
# ============================================================================
def parse_response(text: str):
    text = re.sub(r'^```json\s*|^```\s*|\s*```$', '', text.strip()).strip()
    try:
        start, end = text.find('{'), text.rfind('}') + 1
        if start != -1 and end > start:
            return json.loads(text[start:end])
    except json.JSONDecodeError:
        pass
    return None


def call_gpt(sentence: str, sentence_id: str) -> dict:
    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": sentence}
            ],
            temperature=0.0,
            max_tokens=4096,
            top_p=0.9,
            frequency_penalty=0,
            presence_penalty=0
        )
        logger.info(f"[{sentence_id}] finish_reason: {resp.choices[0].finish_reason}")

        content = resp.choices[0].message.content
        parsed  = parse_response(content) if content else None
        return {"status": "success", **parsed} if parsed else {"status": "parse_failed", "raw": content}

    except Exception as e:
        logger.exception(f"[{sentence_id}] Exception")
        return {"status": "exception", "error": str(e)}

In [ ]:
# ============================================================================
# CELL 3 — Run & Save to Output Excel (Resume-Safe)
# ============================================================================
import os

def extract_fields(errors):
    if not errors:
        return "", "", ""
    spans        = " | ".join(str(e.get("error_span", ""))  for e in errors)
    categories   = " | ".join(str(e.get("category", ""))    for e in errors)
    descriptions = " | ".join(str(e.get("description", "")) for e in errors)
    return spans, categories, descriptions


failed_statuses = {"api_error", "parse_failed", "exception"}

df = pd.read_excel(INPUT_PATH, dtype=str).fillna("")
df = df[df["Incorrect Sentences"].str.strip() != ""].reset_index(drop=True)
logger.info(f"Loaded {len(df)} sentences")

# ── Load existing progress if file exists ──────────────────────────────────
if os.path.exists(OUTPUT_PATH):
    existing_df = pd.read_excel(OUTPUT_PATH, dtype=str).fillna("")
    records     = existing_df.to_dict(orient="records")
    done_sids   = {
        r["Sentence_ID"]
        for r in records
        if r.get("Status") not in failed_statuses and r.get("Status") != ""
    }
    logger.info(f"Resuming — {len(done_sids)} already done, {len(df) - len(done_sids)} remaining.")
else:
    records   = []
    done_sids = set()
    logger.info("No existing file — starting fresh.")

sid_to_idx = {r["Sentence_ID"]: i for i, r in enumerate(records)}

for _, row in df.iterrows():
    sid, sentence = row["Sentence_ID"], row["Incorrect Sentences"]

    if sid in done_sids:
        continue

    result             = call_gpt(sentence, sid)
    errors             = result.get("errors", [])
    spans, cats, descs = extract_fields(errors)

    new_record = {
        "Sentence_ID":         sid,
        "Incorrect Sentences": sentence,
        "Model_has_errors":    result.get("has_errors", ""),
        "Model_error_spans":   spans,
        "Model_categories":    cats,
        "Model_descriptions":  descs,
        "Model_corrected":     result.get("corrected_sentence", ""),
        "Status":              result.get("status", "")
    }

    if sid in sid_to_idx:
        records[sid_to_idx[sid]] = new_record
    else:
        sid_to_idx[sid] = len(records)
        records.append(new_record)

    print(f"\n[{sid}] {sentence}")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print("-" * 60)

    # Save after every row
    pd.DataFrame(records).to_excel(OUTPUT_PATH, index=False)
    time.sleep(1.5)

out_df = pd.DataFrame(records)
logger.info(f"Saved {len(out_df)} rows to {OUTPUT_PATH}")
print(out_df[["Sentence_ID", "Model_has_errors", "Model_error_spans", "Model_categories"]].to_string())

   Sentence_ID Model_has_errors Model_error_spans                 Model_categories
0        SN_01             True                ି଼             Script Normalization
1        SN_02             True                ୋ             Script Normalization
2        SN_03             True               ଓ୍ବ             Script Normalization
3        SN_04             True                 ଡ଼  Spelling & Typographical Errors
4        SN_05             True          ଡ଼ିସେମ୍ବର  Spelling & Typographical Errors
5        SN_06             True           ଖାଁଙ୍କୁ  Spelling & Typographical Errors
6        SN_07             True                େଦ             Script Normalization
7        SN_08             True           ବଢିଥିଲେ               Grammatical Errors
8        SN_09             True            ଆଷାଢ‍଼             Script Normalization
9        SN_10             True            ଫାରାଡ଼େ  Spelling & Typographical Errors
10       SN_11             True                ସହ               Grammatical Errors
11  

In [ ]:
# ============================================================================
# CELL 4 — Retry Failed Rows
# ============================================================================
failed_statuses = {"api_error", "parse_failed", "exception"}
failed_idx = [i for i, r in enumerate(records) if r.get("Status") in failed_statuses]

if not failed_idx:
    print("No failed rows to retry.")
else:
    print(f"Retrying {len(failed_idx)} failed row(s)...\n")
    time.sleep(3)

    for i in failed_idx:
        sid      = records[i]["Sentence_ID"]
        sentence = records[i]["Incorrect Sentences"]

        result = call_gpt(sentence, sid)
        errors = result.get("errors", [])
        spans, cats, descs = extract_fields(errors)

        records[i].update({
            "Model_has_errors":  result.get("has_errors", ""),
            "Model_error_spans": spans,
            "Model_categories":  cats,
            "Model_descriptions":descs,
            "Model_corrected":   result.get("corrected_sentence", ""),
            "Status":            result.get("status", "")
        })

        print(f"\n[{sid}] {sentence}")
        print(json.dumps(result, indent=2, ensure_ascii=False))
        print("-" * 60)
        time.sleep(1.5)

    out_df = pd.DataFrame(records)
    out_df.to_excel(OUTPUT_PATH, index=False)
    logger.info(f"Retry complete. {len(failed_idx)} row(s) updated and saved to {OUTPUT_PATH}")

No failed rows to retry.
